# 🤖 SmolVLA GPU Training & Evaluation Notebook

本筆記本專門為 **Google Colab GPU（T4 / V100 / A100）** 環境設計，提供端到端的 SmolVLA 微調訓練、模擬環境評估與影片生成播放。

⚠️ **請先確認 GPU 已啟用**：
點擊上方選單 **執行階段 (Runtime) -> 變更執行階段類型 (Change runtime type) -> 硬體加速器選擇 T4 GPU**。

In [ ]:
# 1. 驗證 GPU 狀態與環境配置
import torch
import os

if not torch.cuda.is_available():
    raise SystemError("❌ 未檢測到 GPU！請至選單 [執行階段 -> 變更執行階段類型] 切換至 T4 GPU 後再執行。")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print("✅ GPU 檢測成功！")
print("顯卡型號:", gpu_name)
print("顯存大小:", round(vram_gb, 2), "GB")

# 2. 獲取 / 同步最新 LeRobot 倉庫
if not os.path.exists("/content/lerobot"):
    os.system("git clone https://github.com/Child-pi/lerobot.git")
else:
    os.chdir("/content/lerobot")
    os.system("git pull origin main")

%cd /content/lerobot


In [ ]:
# 3. 安裝 LeRobot 及其依賴項
%cd /content/lerobot

print("📦 安裝 LeRobot 與 smolvla 依賴...")
!pip install -e ".[smolvla]"

print("📦 安裝 Aloha 模擬環境與影片編碼套件...")
!pip install dm_env dm_tree glfw lxml "mujoco>=3.0.0,<3.9.0" pyopengl pyparsing
!pip install dm-control --no-deps
!pip install gym-aloha --no-deps
!pip install -q av

print("🎉 環境安裝完成！")


### 🔑 GitHub 認證設定（選填，用於推送到 GitHub 倉庫）
若您希望將訓練結果推回您的 GitHub 倉庫，可在 Colab 左側鑰匙圖示加入 。

In [ ]:
import os
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    if GITHUB_TOKEN:
        GITHUB_USER = "Child-pi"
        REPO_NAME = "lerobot"
        os.system("git config --global user.email nokiya@gmail.com")
        os.system("git config --global user.name Child-pi")
        remote_url = "https://" + GITHUB_USER + ":" + GITHUB_TOKEN + "@github.com/" + GITHUB_USER + "/" + REPO_NAME + ".git"
        os.system("git remote set-url origin " + remote_url)
        print("✅ GitHub 認證設定成功！")
    else:
        print("ℹ️ 未設定 GITHUB_TOKEN，略過 GitHub 認證（不影響本機訓練與評估）。")
except Exception as e:
    print("ℹ️ 略過 GitHub 認證:", e)


### 🚀 啟動 GPU 訓練 SmolVLA
在 Aloha 模擬機械手臂夾取資料集（）上進行 500 步離線微調。
日誌會以**即時串流**方式逐行輸出。

In [ ]:
import os
import shutil
import subprocess
import torch

if os.path.exists("/content/lerobot"):
    os.chdir("/content/lerobot")
    os.system("git pull origin main")

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONUNBUFFERED"] = "1"

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 清理舊輸出目錄避免衝突
output_dir = "outputs/train/smolvla_aloha"
if os.path.exists(output_dir):
    print("🧹 清理先前輸出目錄:", output_dir)
    shutil.rmtree(output_dir)

train_cmd = [
    "lerobot-train",
    "--policy.path=lerobot/smolvla_base",
    "--policy.push_to_hub=false",
    "--dataset.repo_id=lerobot/aloha_sim_insertion_human",
    "--batch_size=2",
    "--steps=500",
    "--save_freq=500",
    "--log_freq=5",
    "--output_dir=" + output_dir,
    "--policy.empty_cameras=2",
    '--rename_map={"observation.images.top": "observation.images.camera1"}',
    "--wandb.enable=false"
]

print("🚀 開始執行 SmolVLA GPU 訓練（即時日誌輸出）...")
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

proc = subprocess.Popen(train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
proc.wait()

if proc.returncode == 0:
    print()
    print("🎉 SmolVLA GPU 訓練順利完成！")
else:
    print()
    print("❌ 訓練異常結束，狀態碼:", proc.returncode)


### 🎬 模擬評估與機械手臂抓取影片錄製
載入剛訓練好的 SmolVLA Checkpoint，在 Aloha 模擬器中執行推論並錄製 .mp4 影片。

In [ ]:
import os
import glob
import subprocess
import torch

if os.path.exists("/content/lerobot"):
    os.chdir("/content/lerobot")

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONUNBUFFERED"] = "1"
if torch.cuda.is_available():
    torch.cuda.empty_cache()

candidate_patterns = [
    "/content/lerobot/outputs/train/**/pretrained_model",
    "outputs/train/**/pretrained_model",
    "/content/lerobot/outputs/train/**/checkpoints/*",
]
checkpoints = []
for pat in candidate_patterns:
    checkpoints.extend(glob.glob(pat, recursive=True))
checkpoints = [p for p in checkpoints if os.path.isdir(p) and (os.path.exists(os.path.join(p, "config.json")) or "pretrained_model" in p)]
checkpoints = list(dict.fromkeys(checkpoints))

if not checkpoints:
    print("❌ 找不到訓練好的模型，請確認上一格訓練是否完成。")
else:
    ckpt_path = checkpoints[0]
    print("✅ 找到評估目標模型:", ckpt_path)
    print("🎬 開始執行模擬評估並錄製影片...")
    
    eval_cmd = [
        "lerobot-eval",
        "--policy.path=" + str(ckpt_path),
        "--env.type=aloha",
        "--eval.n_episodes=1",
        "--eval.batch_size=1",
        "--eval.use_async_envs=false",
        '--rename_map={"observation.images.top": "observation.images.camera1"}',
        "--output_dir=/content/lerobot/outputs/eval/smolvla_aloha"
    ]
    
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    eval_proc = subprocess.Popen(eval_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in iter(eval_proc.stdout.readline, ""):
        print(line, end="", flush=True)
    eval_proc.wait()
    
    if eval_proc.returncode == 0:
        print()
        print("✅ 模擬評估與錄影完成！請執行下一格播放影片。")
    else:
        print()
        print("⚠️ 評估指令結束，狀態碼:", eval_proc.returncode)


### 📺 播放機械手臂模擬抓取影片

In [ ]:
from IPython.display import Video, display
import glob
import os

if os.path.exists("/content/lerobot"):
    os.chdir("/content/lerobot")

search_patterns = [
    "/content/lerobot/outputs/eval/**/videos/*.mp4",
    "/content/lerobot/outputs/**/videos/*.mp4",
    "outputs/eval/**/videos/*.mp4",
    "outputs/**/videos/*.mp4",
    "/content/**/*.mp4"
]
all_videos = []
for pat in search_patterns:
    all_videos.extend(glob.glob(pat, recursive=True))
all_videos = list(dict.fromkeys(all_videos))
valid_videos = [v for v in all_videos if os.path.isfile(v)]

if valid_videos:
    valid_videos.sort(key=os.path.getmtime)
    latest_video = valid_videos[-1]
    print("🎉 成功找到機械手臂評估影片:", latest_video)
    print("檔案大小:", round(os.path.getsize(latest_video) / 1024, 1), "KB")
    display(Video(latest_video, embed=True, width=640))
else:
    print("⚠️ 尚未找到錄製的 .mp4 影片！請先確認上方評估儲存格是否已執行完畢。")
